In [1]:
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
) 
import torch
from qwen_vl_utils import process_vision_info

/home/mayflower/.local/share/mamba/envs/EHA/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/mayflower/.local/share/mamba/envs/EHA/lib/python3.10/site-packages/transformers/utils/hub.py:109: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
def load_model(model_id):
    """Load the model and processor."""
    min_pixels = 256 * 28 * 28
    max_pixels = 512 * 28 * 28
    processor = AutoProcessor.from_pretrained(
        model_id, trust_remote_code=True, min_pixels=min_pixels, max_pixels=max_pixels
    )

    model = AutoModelForImageTextToText.from_pretrained(
        model_id,
        dtype=torch.bfloat16,
        trust_remote_code=True,
        #attn_implementation="flash_attention_2",
        device_map="auto",
    )

    model.eval()
    return model, processor

In [3]:
def prepare_inputs(model, processor, image_paths, questions):
    """Build model-ready inputs from batches of images + text."""
    
    # Ensure inputs are lists for batch processing
    if isinstance(image_paths, str):
        image_paths = [image_paths]
    if isinstance(questions, str):
        questions = [questions]
    
    # Validate batch sizes match
    batch_size = len(image_paths)
    if len(questions) != batch_size:
        raise ValueError(f"Batch size mismatch: {len(image_paths)} images vs {len(questions)} questions")
    
    # Build messages for each item in the batch
    messages = []
    for img_path, question in zip(image_paths, questions):
        messages.append({
            "role": "user",
            "content": [
                {"type": "image", "image": img_path},
                {"type": "text", "text": question},
            ],
        })
    
    # Apply chat template to all messages
    texts = [
        processor.apply_chat_template([m], tokenize=False, add_generation_prompt=True)
        for m in messages
    ]
    
    # Process vision info for all messages
    image_inputs, video_inputs = process_vision_info(messages)
    
    # Process all inputs together as a batch
    inputs = processor(
        text=texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    
    return inputs.to(model.device)

In [4]:
model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
model, processor = load_model(model_id)

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


🚨 `pixel_values_cd` is part of Qwen2_5_VLForConditionalGeneration.forward's signature, but not documented. Make sure to add it to the docstring of the function in /home/mayflower/.local/share/mamba/envs/EHA/lib/python3.10/site-packages/transformers/models/qwen2_5_vl/modeling_qwen2_5_vl.py.


Loading checkpoint shards: 100%|█████████████████████████████████████████████████████████████| 2/2 [00:02<00:00,  1.28s/it]


In [ ]:
# Single input (backward compatible)
inputs = prepare_inputs(model, processor, "../datasets/coco2014/val2014/COCO_val2014_000000000042.jpg", "What's in this image?")

In [ ]:
# Then generate
outputs = model.generate(**inputs, max_new_tokens=64)

In [ ]:
responses = processor.batch_decode(outputs, skip_special_tokens=True)
print(responses)

In [5]:
# Batch input
inputs = prepare_inputs(
    model, 
    processor, 
    ["../datasets/coco2014/val2014/COCO_val2014_000000000042.jpg", "../datasets/coco2014/val2014/COCO_val2014_000000000073.jpg", "../datasets/coco2014/val2014/COCO_val2014_000000000074.jpg"],
    ["Describe this", "What color is this?", "Count the objects"]
)

# Then generate
outputs = model.generate(**inputs, max_new_tokens=128)
responses = processor.batch_decode(outputs, skip_special_tokens=True)


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


In [6]:
print(responses)

['system\nYou are a helpful assistant.\nuser\nDescribe this\nassistant\n', 'system\nYou are a helpful assistant.\nuser\nWhat color is this?\nassistant\nThe motorcycle in the picture is primarily black with some silver and blue accents. The license plate reads "SV-6260."', 'system\nYou are a helpful assistant.\nuser\nCount the objects\nassistant\n\n']
